In [1]:
import numpy as np

# ---------------------------------------------------------
# Step 1: Input Image (5x5 grayscale)
# ---------------------------------------------------------
image = np.array([
    [3, 1, 0, 2, 1],
    [2, 0, 1, 3, 0],
    [4, 1, 0, 1, 2],
    [1, 3, 2, 0, 1],
    [0, 2, 1, 4, 2]
], dtype=np.float32)

# ---------------------------------------------------------
# Step 2: Convolution (3x3 Kernel, stride=1, no padding)
# ---------------------------------------------------------
kernel = np.array([
    [1,  0, -1],
    [1,  0, -1],
    [1,  0, -1]
], dtype=np.float32)

h_out = image.shape[0] - kernel.shape[0] + 1  # 5 - 3 + 1 = 3
w_out = image.shape[1] - kernel.shape[1] + 1  # 5 - 3 + 1 = 3
feature_map = np.zeros((h_out, w_out), dtype=np.float32)

for i in range(h_out):
    for j in range(w_out):
        region = image[i:i+3, j:j+3]
        feature_map[i, j] = np.sum(region * kernel)

print("Step 2: Feature Map:\n", feature_map)

# ---------------------------------------------------------
# Step 3: ReLU Activation (f(x) = max(0, x))
# ---------------------------------------------------------
relu_out = np.maximum(0, feature_map)
print("\nStep 3: After ReLU:\n", relu_out)

# ---------------------------------------------------------
# Step 4: Max Pooling (2x2 pool window, stride=1)
# ---------------------------------------------------------
pool_h = relu_out.shape[0] - 2 + 1  # 3 - 2 + 1 = 2
pool_w = relu_out.shape[1] - 2 + 1  # 3 - 2 + 1 = 2
pooled = np.zeros((pool_h, pool_w), dtype=np.float32)

for i in range(pool_h):
    for j in range(pool_w):
        window = relu_out[i:i+2, j:j+2]
        pooled[i, j] = np.max(window)

print("\nStep 4: Pooled (Max Pooling 2x2):\n", pooled)

# ---------------------------------------------------------
# Step 5: Flatten + Fully Connected Layer
# ---------------------------------------------------------
flattened = pooled.flatten()  # Vector: [8., 0., 4., 1.]

weights = np.array([
    [0.3, 0.1, 0.2, 0.1],  # Cat
    [0.1, 0.2, 0.1, 0.3],  # Dog
    [0.0, 0.4, 0.1, 0.2]   # Bird
], dtype=np.float32)

logits = np.dot(weights, flattened)
print("\nStep 5: Flattened Vector:", flattened)
print("Step 5: Logits (z):", logits)

# ---------------------------------------------------------
# Step 6: Softmax Output
# ---------------------------------------------------------
exp_z = np.exp(logits)
probabilities = exp_z / np.sum(exp_z)

classes = ["Cat", "Dog", "Bird"]
print("\nStep 6: Softmax Probabilities:")
for label, prob in zip(classes, probabilities):
    print(f"  {label}: {prob * 100:.1f}%")

Step 2: Feature Map:
 [[ 8. -4. -2.]
 [ 4.  0.  0.]
 [ 2.  1. -2.]]

Step 3: After ReLU:
 [[8. 0. 0.]
 [4. 0. 0.]
 [2. 1. 0.]]

Step 4: Pooled (Max Pooling 2x2):
 [[8. 0.]
 [4. 1.]]

Step 5: Flattened Vector: [8. 0. 4. 1.]
Step 5: Logits (z): [3.3000002 1.5       0.6      ]

Step 6: Softmax Probabilities:
  Cat: 81.1%
  Dog: 13.4%
  Bird: 5.5%


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input channel, 1 output channel, 3x3 kernel, no bias, no padding
        self.conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=1)
        self.fc = nn.Linear(in_features=4, out_features=3, bias=False)

        # Initialize weights with the exact values from the slides
        with torch.no_grad():
            self.conv.weight = nn.Parameter(
                torch.tensor([[[[1.,  0., -1.],
                                [1.,  0., -1.],
                                [1.,  0., -1.]]]])
            )
            self.fc.weight = nn.Parameter(
                torch.tensor([[0.3, 0.1, 0.2, 0.1],   # Cat
                              [0.1, 0.2, 0.1, 0.3],   # Dog
                              [0.0, 0.4, 0.1, 0.2]])  # Bird
            )

    def forward(self, x):
        x = self.conv(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        return logits, probs

# Input shape: (batch_size=1, channels=1, height=5, width=5)
x = torch.tensor([[[
    [3., 1., 0., 2., 1.],
    [2., 0., 1., 3., 0.],
    [4., 1., 0., 1., 2.],
    [1., 3., 2., 0., 1.],
    [0., 2., 1., 4., 2.]
]]])

model = SimpleCNN()
logits, probs = model(x)

classes = ["Cat", "Dog", "Bird"]
for label, prob in zip(classes, probs[0]):
    print(f"{label}: {prob.item() * 100:.1f}%")

In [3]:
import numpy as np

# 1. Input Image
x = np.array([
    [3, 1, 0, 2, 1],
    [2, 0, 1, 3, 0],
    [4, 1, 0, 1, 2],
    [1, 3, 2, 0, 1],
    [0, 2, 1, 4, 2]
], dtype=np.float32)

# 2. Convolution (3x3 Kernel, stride 1)
kernel = np.array([
    [1,  0, -1],
    [1,  0, -1],
    [1,  0, -1]
], dtype=np.float32)

h_out, w_out = x.shape[0] - 3 + 1, x.shape[1] - 3 + 1
conv_out = np.zeros((h_out, w_out))
for i in range(h_out):
    for j in range(w_out):
        conv_out[i, j] = np.sum(x[i:i+3, j:j+3] * kernel)

# 3. ReLU
relu_out = np.maximum(0, conv_out)

# 4. Max Pooling (2x2, stride 1)
pool_h, pool_w = relu_out.shape[0] - 2 + 1, relu_out.shape[1] - 2 + 1
pooled = np.zeros((pool_h, pool_w))
for i in range(pool_h):
    for j in range(pool_w):
        pooled[i, j] = np.max(relu_out[i:i+2, j:j+2])

# 5. Flatten & Linear Layer
flattened = pooled.flatten()
weights = np.array([
    [0.3, 0.1, 0.2, 0.1],  # Cat
    [0.1, 0.2, 0.1, 0.3],  # Dog
    [0.0, 0.4, 0.1, 0.2]   # Bird
])
logits = np.dot(weights, flattened)

# 6. Softmax
exp_z = np.exp(logits)
probs = exp_z / np.sum(exp_z)

classes = ["Cat", "Dog", "Bird"]
for label, p in zip(classes, probs):
    print(f"{label}: {p * 100:.1f}%")

Cat: 81.1%
Dog: 13.4%
Bird: 5.5%


In [4]:
import numpy as np

# -------------------------------------------------------------------------
# Step 1: Input Image (5x5 grayscale)
# -------------------------------------------------------------------------
image = np.array([
    [3, 1, 0, 2, 1],
    [2, 0, 1, 3, 0],
    [4, 1, 0, 1, 2],
    [1, 3, 2, 0, 1],
    [0, 2, 1, 4, 2]
], dtype=np.float32)

print("--- Step 1: Input Image ---")
print(image)

# -------------------------------------------------------------------------
# Step 2: Convolution Layer (3x3 Kernel, Stride 1, No Padding)
# -------------------------------------------------------------------------
kernel = np.array([
    [1,  0, -1],
    [1,  0, -1],
    [1,  0, -1]
], dtype=np.float32)

out_h = image.shape[0] - kernel.shape[0] + 1  # 5 - 3 + 1 = 3
out_w = image.shape[1] - kernel.shape[1] + 1  # 5 - 3 + 1 = 3
feature_map = np.zeros((out_h, out_w), dtype=np.float32)

for r in range(out_h):
    for c in range(out_w):
        window = image[r:r+3, c:c+3]
        feature_map[r, c] = np.sum(window * kernel)

print("\n--- Step 2: Feature Map ---")
print(feature_map)

# -------------------------------------------------------------------------
# Step 3: ReLU Activation (f(x) = max(0, x))
# -------------------------------------------------------------------------
relu_out = np.maximum(0, feature_map)

print("\n--- Step 3: ReLU Output ---")
print(relu_out)

# -------------------------------------------------------------------------
# Step 4: Max Pooling (2x2 Window, Stride 1)
# -------------------------------------------------------------------------
pool_h = relu_out.shape[0] - 2 + 1  # 3 - 2 + 1 = 2
pool_w = relu_out.shape[1] - 2 + 1  # 3 - 2 + 1 = 2
pooled = np.zeros((pool_h, pool_w), dtype=np.float32)

for r in range(pool_h):
    for c in range(pool_w):
        window = relu_out[r:r+2, c:c+2]
        pooled[r, c] = np.max(window)

print("\n--- Step 4: Max Pooling (2x2) ---")
print(pooled)

# -------------------------------------------------------------------------
# Step 5: Flatten + Fully Connected Layer
# -------------------------------------------------------------------------
flattened = pooled.flatten()  # [8.0, 0.0, 4.0, 1.0]

weights = np.array([
    [0.3, 0.1, 0.2, 0.1],  # Cat
    [0.1, 0.2, 0.1, 0.3],  # Dog
    [0.0, 0.4, 0.1, 0.2]   # Bird
], dtype=np.float32)

logits = np.dot(weights, flattened)

print("\n--- Step 5: Flatten & Logits ---")
print("Flattened Vector:", flattened)
print("Logits (z):", logits)

# -------------------------------------------------------------------------
# Step 6: Softmax Output
# -------------------------------------------------------------------------
exp_z = np.exp(logits)
probabilities = exp_z / np.sum(exp_z)

classes = ["Cat", "Dog", "Bird"]
print("\n--- Step 6: Softmax Probabilities ---")
for label, p in zip(classes, probabilities):
    print(f"{label}: {p * 100:.1f}%")

best_idx = np.argmax(probabilities)
print(f"\nFinal Prediction: {classes[best_idx]} ({probabilities[best_idx] * 100:.1f}%)")

--- Step 1: Input Image ---
[[3. 1. 0. 2. 1.]
 [2. 0. 1. 3. 0.]
 [4. 1. 0. 1. 2.]
 [1. 3. 2. 0. 1.]
 [0. 2. 1. 4. 2.]]

--- Step 2: Feature Map ---
[[ 8. -4. -2.]
 [ 4.  0.  0.]
 [ 2.  1. -2.]]

--- Step 3: ReLU Output ---
[[8. 0. 0.]
 [4. 0. 0.]
 [2. 1. 0.]]

--- Step 4: Max Pooling (2x2) ---
[[8. 0.]
 [4. 1.]]

--- Step 5: Flatten & Logits ---
Flattened Vector: [8. 0. 4. 1.]
Logits (z): [3.3000002 1.5       0.6      ]

--- Step 6: Softmax Probabilities ---
Cat: 81.1%
Dog: 13.4%
Bird: 5.5%

Final Prediction: Cat (81.1%)
